# EXP-010 (dataset baru): Temporal Kinematic Fall Velocity

Mengulang EXP-010 dengan dataset gabungan buatan Laila (**UR Fall + MCF dari UMontreal**, 8.645 gambar).
Dataset lama (UR Fall, 1.498 gambar) **tidak** dipakai di notebook ini.

**Cara pakai di Kaggle**
1. Zip folder `Data Baru` (isi: `images/`, `labels/`, `data.yaml`) lalu upload sebagai Kaggle Dataset.
2. (Opsional) upload juga `yolo11n-pose.pt` ke dataset yang sama. Kalau tidak ada, notebook mengunduhnya (butuh **Internet: On**).
3. Aktifkan **GPU** (Settings > Accelerator), lalu **Run All**.
4. Unduh `exp010_new_outputs.zip` dari tab *Output* dan kirim ke Raja untuk dianalisis.

**Beda dengan EXP-010 lama (dan kenapa)**
- Nama sequence di dataset baru berbeda: `urfall_fall-26_f0001.jpg`, `mcf_s19_c3_f00877.jpg`. Pengelompokan diperbaiki (`urfall_<seq>` dan `mcf_<skenario>_<kamera>`).
- Jarak antar frame **tidak seragam** (UR Fall: 5; MCF: 2, 10, 30). Jadi kecepatan dihitung dua cara: `Vy_index` (sama seperti EXP-010 lama, per urutan gambar) dan `Vy_frame` (per frame video asli, memakai nomor frame).
- Hasil dilaporkan **per sumber** (UR Fall vs MCF) dan **per kamera** MCF, karena MCF punya 8 sudut kamera per skenario.
- Label sequence: UR Fall dari nama (`fall-*` = jatuh). MCF dari label frame: sequence dianggap jatuh jika ada frame berkelas `lying_on_ground`.

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import json, math, re, time, shutil, zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
# ---- Konfigurasi -------------------------------------------------------
CONF = 0.25              # confidence deteksi (sama dengan EXP-010 lama)
K_WINDOW = 2             # jendela selisih Vy (sama dengan EXP-010 lama)
ANGLE_LYING = 25.0       # ambang sudut 'rebah' (sama dengan EXP-010 lama)
LEGACY_VEL_CAP = 0.035   # batas atas threshold Vy_index (sama dengan EXP-010 lama)
BATCH = 32
WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("exp010_out")
WORK.mkdir(parents=True, exist_ok=True)


def find_dataset_root():
    bases = [Path("/kaggle/input"), Path("."), Path("Data Baru"), Path("../Data Baru")]
    for b in bases:
        if not b.exists():
            continue
        for cand in [b, *b.rglob("*")]:
            if cand.is_dir() and (cand / "images" / "test").is_dir() and (cand / "labels" / "test").is_dir():
                return cand
    raise FileNotFoundError("Folder dataset (berisi images/ dan labels/) tidak ditemukan. Cek upload Kaggle Dataset.")


def find_model():
    for p in list(Path("/kaggle/input").rglob("yolo11n-pose.pt")) if Path("/kaggle/input").exists() else []:
        return str(p)
    return "yolo11n-pose.pt"   # diunduh otomatis (butuh Internet)


ROOT = find_dataset_root()
MODEL_PATH = find_model()
print("Dataset :", ROOT)
print("Model   :", MODEL_PATH)
for sp in ("train", "val", "test"):
    print(f"  {sp:5s}: {len(list((ROOT/'images'/sp).glob('*.jpg')))} gambar")

In [ ]:
def parse_name(stem):
    """Kembalikan (sumber, kunci_sequence, kamera, nomor_frame) dari nama file."""
    m = re.match(r"mcf_(s\d+)_(c\d+)_f(\d+)", stem)
    if m:
        return "mcf", f"mcf_{m.group(1)}_{m.group(2)}", m.group(2), int(m.group(3))
    m = re.match(r"urfall_((?:fall|adl)-\d+)_f(\d+)", stem)
    if m:
        return "urfall", f"urfall_{m.group(1)}", "cam0", int(m.group(2))
    return None


def spine_angle(k):
    sx, sy = (k[5][0] + k[6][0]) / 2, (k[5][1] + k[6][1]) / 2
    hx, hy = (k[11][0] + k[12][0]) / 2, (k[11][1] + k[12][1]) / 2
    return math.degrees(math.atan2(abs(sy - hy), abs(sx - hx)))


def read_gt(lbl):
    t = lbl.read_text().split()
    return int(t[0]) if t else -1


def extract_split(model, split):
    """Fitur per-frame untuk satu split. Satu baris = satu gambar."""
    paths = sorted((ROOT / "images" / split).glob("*.jpg"))
    rows = []
    for i in range(0, len(paths), BATCH):
        chunk = paths[i:i + BATCH]
        results = model([str(p) for p in chunk], verbose=False, conf=CONF)
        for p, r in zip(chunk, results):
            info = parse_name(p.stem)
            if info is None:
                continue
            src, seq, cam, fno = info
            row = dict(split=split, source=src, seq=seq, camera=cam, frame=fno, img=p.name,
                       gt=read_gt(ROOT / "labels" / split / (p.stem + ".txt")),
                       detected=0, kpt_ok=0, angle=np.nan, hip_y=np.nan, ar=np.nan, conf=0.0)
            if r.boxes is not None and len(r.boxes) > 0:
                b = int(r.boxes.conf.argmax().item())
                x1, y1, x2, y2 = r.boxes.xyxy[b].tolist()
                h = max(1.0, y2 - y1)
                k = r.keypoints.data[b].cpu().numpy()
                row.update(detected=1, ar=(x2 - x1) / h, conf=float(r.boxes.conf[b].item()))
                img_h = r.orig_shape[0]
                if (k[5][2] + k[6][2]) / 2 > 0.25 and (k[11][2] + k[12][2]) / 2 > 0.25:
                    row.update(kpt_ok=1, angle=spine_angle(k), hip_y=((k[11][1] + k[12][1]) / 2) / img_h)
                else:   # fallback bentuk kotak, sama seperti EXP-010 lama
                    ar = row["ar"]
                    row.update(angle=90.0 if ar < 0.8 else (20.0 if ar > 1.3 else 50.0),
                               hip_y=((y1 + y2) / 2) / img_h)
            rows.append(row)
    return pd.DataFrame(rows)


FEAT_CSV = WORK / "exp010_new_features.csv"
if FEAT_CSV.exists():
    feats = pd.read_csv(FEAT_CSV)
    print("Memakai cache:", FEAT_CSV, len(feats))
else:
    model = YOLO(MODEL_PATH)
    t0 = time.time()
    feats = pd.concat([extract_split(model, s) for s in ("val", "test")], ignore_index=True)
    feats.to_csv(FEAT_CSV, index=False)
    print(f"Ekstraksi selesai: {len(feats)} gambar dalam {time.time()-t0:.0f} dtk")

print(feats.groupby(["split", "source"]).agg(n=("img", "count"), terdeteksi=("detected", "mean"),
                                                 sendi_ok=("kpt_ok", "mean")).round(3))

## Dinamika per sequence
Sama dengan EXP-010 lama: bila orang tidak terdeteksi, nilai frame sebelumnya dipakai ulang (atau 90 derajat / 0.5 untuk frame pertama).

In [ ]:
def sequence_dynamics(g):
    g = g.sort_values("frame").reset_index(drop=True)
    ang = g["angle"].to_numpy(dtype=float).copy()
    hip = g["hip_y"].to_numpy(dtype=float).copy()
    for i in range(len(g)):                       # isi frame tanpa deteksi dengan nilai sebelumnya
        if np.isnan(ang[i]):
            ang[i] = 90.0 if i == 0 else ang[i - 1]
            hip[i] = 0.5 if i == 0 else hip[i - 1]
    fr = g["frame"].to_numpy(dtype=float)
    v_idx = np.zeros(len(g))
    v_frm = np.zeros(len(g))
    for t in range(K_WINDOW, len(g)):
        dy = hip[t] - hip[t - K_WINDOW]
        v_idx[t] = dy / K_WINDOW
        dt = fr[t] - fr[t - K_WINDOW]
        v_frm[t] = dy / dt if dt > 0 else 0.0
    return dict(n_frames=len(g), min_angle=float(ang.min()), max_vy_index=float(v_idx.max()),
                max_vy_frame=float(v_frm.max()), angles=ang, vy_index=v_idx, vy_frame=v_frm,
                frames=fr, gt=g["gt"].to_numpy())


def sequence_label(seq, gt_frames):
    if seq.startswith("urfall_"):
        return int(seq.startswith("urfall_fall"))
    return int((gt_frames == 2).any())          # MCF: ada frame lying_on_ground


dyn = {}
rows = []
for (split, seq), g in feats.groupby(["split", "seq"]):
    d = sequence_dynamics(g)
    d["label"] = sequence_label(seq, g["gt"].to_numpy())
    dyn[(split, seq)] = d
    rows.append(dict(split=split, seq=seq, source=g["source"].iloc[0], camera=g["camera"].iloc[0],
                     label=d["label"], n_frames=d["n_frames"], min_angle=d["min_angle"],
                     max_vy_index=d["max_vy_index"], max_vy_frame=d["max_vy_frame"]))
seqdf = pd.DataFrame(rows)
print(seqdf.groupby(["split", "source", "label"]).size().unstack(fill_value=0).rename(columns={0: "ADL", 1: "FALL"}))
print("\nSequence pendek (<=4 frame, Vy tidak bermakna):", int((seqdf.n_frames <= 4).sum()))

## Kalibrasi threshold di Validation Set, evaluasi di Test Set
Rumus identik dengan EXP-010 lama: threshold = titik tengah rata-rata `max Vy` FALL dan ADL di val.
Untuk `Vy_index` dibatasi maksimum 0.035 (seperti dulu). Catatan: val untuk ADL hanya berasal dari UR Fall (MCF val semuanya sequence jatuh).

In [ ]:
val = seqdf[seqdf.split == "val"]
test = seqdf[seqdf.split == "test"].copy()


def midpoint(col):
    f, a = val[val.label == 1][col], val[val.label == 0][col]
    return (f.mean() + a.mean()) / 2.0, f.mean(), a.mean()


thr_idx_raw, f_idx, a_idx = midpoint("max_vy_index")
thr_idx = min(thr_idx_raw, LEGACY_VEL_CAP)
thr_frm, f_frm, a_frm = midpoint("max_vy_frame")
print(f"Vy_index : rata2 FALL={f_idx:.4f}  ADL={a_idx:.4f}  titik tengah={thr_idx_raw:.4f}  dipakai={thr_idx:.4f}")
print(f"Vy_frame : rata2 FALL={f_frm:.5f}  ADL={a_frm:.5f}  dipakai={thr_frm:.5f}")

test["pred_static"] = (test.min_angle < ANGLE_LYING).astype(int)
test["pred_temp_index"] = ((test.min_angle < ANGLE_LYING) & (test.max_vy_index >= thr_idx)).astype(int)
test["pred_temp_frame"] = ((test.min_angle < ANGLE_LYING) & (test.max_vy_frame >= thr_frm)).astype(int)


def metrics(df, col):
    tp = int(((df.label == 1) & (df[col] == 1)).sum()); fp = int(((df.label == 0) & (df[col] == 1)).sum())
    fn = int(((df.label == 1) & (df[col] == 0)).sum()); tn = int(((df.label == 0) & (df[col] == 0)).sum())
    n = tp + fp + fn + tn
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return dict(n=n, FALL=tp + fn, ADL=tn + fp, acc=(tp + tn) / n if n else 0.0, precision=p, recall=r,
                f1=2 * p * r / (p + r) if p + r else 0.0, TP=tp, FP=fp, FN=fn, TN=tn)


report = []
groups = {"SEMUA": test, "urfall": test[test.source == "urfall"], "mcf": test[test.source == "mcf"]}
for gname, df in groups.items():
    for col, label in (("pred_static", "static angle < 25"), ("pred_temp_index", "angle + Vy_index"),
                       ("pred_temp_frame", "angle + Vy_frame")):
        report.append(dict(kelompok=gname, metode=label, **metrics(df, col)))
report = pd.DataFrame(report)
pd.set_option("display.width", 200)
print(report.round(3).to_string(index=False))

In [ ]:
# Rincian per kamera MCF dan per sequence (untuk melihat sudut pandang mana yang gagal)
mcf_t = test[test.source == "mcf"].copy()
cam_tbl = mcf_t.groupby("camera").apply(lambda d: pd.Series(dict(
    n=len(d), FALL=int(d.label.sum()),
    recall_static=((d.pred_static == 1) & (d.label == 1)).sum() / max(1, d.label.sum()),
    recall_temp_index=((d.pred_temp_index == 1) & (d.label == 1)).sum() / max(1, d.label.sum()),
    FP_static=int(((d.pred_static == 1) & (d.label == 0)).sum())))).round(3)
print(cam_tbl)
print()
print(test.sort_values(["source", "seq"])[["seq", "label", "n_frames", "min_angle", "max_vy_index", "max_vy_frame",
                                          "pred_static", "pred_temp_index", "pred_temp_frame"]].round(4).to_string(index=False))

In [ ]:
# Kurva contoh: UR Fall (fall vs adl) dan satu skenario MCF dari beberapa kamera
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
ex_fall = next((k for k in dyn if k[0] == "test" and k[1] == "urfall_fall-26"), None)
ex_adl = next((k for k in dyn if k[0] == "test" and k[1] == "urfall_adl-35"), None)
for col, key, color in ((0, ex_fall, "crimson"), (1, ex_adl, "seagreen")):
    if key is None:
        continue
    d = dyn[key]
    ax[0, col].plot(d["frames"], d["angles"], color=color, lw=2)
    ax[0, col].axhline(ANGLE_LYING, color="k", ls="--", lw=1)
    ax[0, col].set_title(f"{key[1]} - sudut tulang belakang"); ax[0, col].set_ylabel("derajat")
    ax[1, col].plot(d["frames"], d["vy_frame"], color=color, lw=2)
    ax[1, col].axhline(thr_frm, color="b", ls="--", lw=1, label=f"threshold {thr_frm:.4f}")
    ax[1, col].set_title(f"{key[1]} - Vy per frame asli"); ax[1, col].set_xlabel("nomor frame"); ax[1, col].legend()
for a in ax.ravel():
    a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(WORK / "exp010_new_urfall_curves.png", dpi=150); plt.show()

mcf_fall = sorted(k for k in dyn if k[0] == "test" and k[1].startswith("mcf_s19_"))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for k in mcf_fall:
    d = dyn[k]
    ax[0].plot(d["frames"], d["angles"], lw=1.5, label=k[1].split("_")[-1])
    ax[1].plot(d["frames"], d["vy_frame"], lw=1.5, label=k[1].split("_")[-1])
ax[0].axhline(ANGLE_LYING, color="k", ls="--"); ax[0].set_title("MCF s19: sudut dari 8 kamera"); ax[0].legend(ncol=4, fontsize=7)
ax[1].axhline(thr_frm, color="b", ls="--"); ax[1].set_title("MCF s19: Vy per frame dari 8 kamera")
for a in ax:
    a.grid(alpha=0.3); a.set_xlabel("nomor frame")
plt.tight_layout(); plt.savefig(WORK / "exp010_new_mcf_curves.png", dpi=150); plt.show()

In [ ]:
# Simpan semua keluaran, lalu zip untuk diunduh
seqdf.to_csv(WORK / "exp010_new_sequence_summary.csv", index=False)
test.to_csv(WORK / "exp010_new_test_predictions.csv", index=False)
report.to_csv(WORK / "exp010_new_report.csv", index=False)
result = dict(
    experiment="EXP-010", dataset="Data Baru (UR Fall + MCF, buatan Laila)", model=MODEL_PATH,
    params=dict(conf=CONF, k_window=K_WINDOW, angle_lying=ANGLE_LYING),
    thresholds=dict(vy_index=float(thr_idx), vy_index_uncapped=float(thr_idx_raw), vy_frame=float(thr_frm)),
    val_vy=dict(index_fall=float(f_idx), index_adl=float(a_idx), frame_fall=float(f_frm), frame_adl=float(a_frm)),
    report=report.round(4).to_dict(orient="records"),
)
(WORK / "exp010_new_results.json").write_text(json.dumps(result, indent=2))

zip_path = WORK / "exp010_new_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("exp010_new_features.csv", "exp010_new_sequence_summary.csv", "exp010_new_test_predictions.csv",
                 "exp010_new_report.csv", "exp010_new_results.json", "exp010_new_urfall_curves.png",
                 "exp010_new_mcf_curves.png"):
        if (WORK / name).exists():
            z.write(WORK / name, name)
print("Selesai. Unduh:", zip_path)